Concurrency, The GIL & Shared Memory
Core Mechanics & Theory
To parallelize RL rollout environments, stream data to GPU buffers, and build multi-core pipelines, you must understand how CPython handles execution at the OS thread and process levels.

First: what is the GIL?

GIL = Global Interpreter Lock.

In standard CPython, the GIL means:

At a given moment, only one thread can execute Python bytecode within a Python interpreter.

Suppose you have:

def compute():
    total = 0
    for i in range(10_000_000):
        total += i

This is CPU-bound Python work.

If you create:

Thread 1 → compute()
Thread 2 → compute()

you might expect:

CPU Core 1 → Thread 1
CPU Core 2 → Thread 2

But for Python bytecode, the GIL prevents both threads from executing Python bytecode simultaneously.

Conceptually:

Thread 1: ████████     ████████
Thread 2:         ████████     ████████
                 ↑
              GIL moves

So you don't get true parallel execution of the Python bytecode.

2. Why does Python have the GIL?

This is where the technical side matters.

CPython uses reference counting for memory management.

An object has a reference count conceptually like:

object
  │
  └── refcount = 3

When another variable references it:

refcount += 1

When a reference disappears:

refcount -= 1

These operations need to be safely coordinated between threads.

The GIL historically provides a simple mechanism to protect CPython's interpreter state and memory-management machinery.

So:

GIL isn't fundamentally a "threading feature"; it's a CPython implementation mechanism that restricts simultaneous execution of Python bytecode.

Also, one important modern nuance: free-threaded CPython builds exist, so "Python always has a GIL" is no longer universally true. But for the standard CPython setup you're likely using, the GIL model in your exercise is the right one to learn.

Threading vs multiprocessing

This is the most important distinction.

Thread
threading.Thread(...)

Threads live inside the same process.

So:

Process
│
├── Thread 1
├── Thread 2
└── Thread 3

They share the process's memory.

That's convenient:

data = [1, 2, 3]

Both threads can access data.

But shared memory means you can get:

Race conditions

and may need synchronization primitives such as:

Lock
Process
multiprocessing.Process(...)

creates another OS process.

Conceptually:

Process 1
    Memory A


Process 2
    Memory B

They don't normally share Python objects directly.

That's why processes can execute Python bytecode truly in parallel on multiple CPU cores.

4. When should you use each?

The simple rule:

CPU-bound Python
      ↓
multiprocessing


I/O-bound work
      ↓
threading / asyncio

For example:

CPU-bound
huge_python_calculation()

Processes are usually the better choice.

I/O-bound
download_file()
wait_for_network()
read_from_socket()

Threads can be useful because while one thread is waiting for I/O, another can run.

GIL benchmark

Let's build exactly what your exercise asks.

In [1]:
import time
import threading
import multiprocessing


def compute_heavy(n: int):
    return sum(i * i for i in range(n))

In [3]:
N = 10_000_000

start = time.perf_counter()

compute_heavy(N)
compute_heavy(N)

elapsed = time.perf_counter() - start

print(f"Sequential: {elapsed:.2f}s")

Sequential: 1.35s


In [4]:
def run_thread():
    compute_heavy(N)
t1 = threading.Thread(target=run_thread)
t2 = threading.Thread(target=run_thread)

start = time.perf_counter()

t1.start()
t2.start()

t1.join()
t2.join()

elapsed = time.perf_counter() - start

print(f"Threads: {elapsed:.2f}s")

Threads: 1.54s


What do we expect?

You might expect:

Sequential:
Task A ██████████
Task B           ██████████


Threads:
Task A ██████████
Task B ██████████

and therefore threads should be twice as fast.

But for CPU-bound Python:

Threads:
A ███  ███  ███  ███
B   ███  ███  ███  ███

The threads take turns executing Python bytecode because of the GIL.

Therefore you might see something roughly like:

Sequential: 1.5 sec
Threads:    1.6 sec

or even worse depending on your machine.

In [6]:
def run_process():
    compute_heavy(N)

In [7]:
p1 = multiprocessing.Process(target=run_process)
p2 = multiprocessing.Process(target=run_process)

start = time.perf_counter()

p1.start()
p2.start()

p1.join()
p2.join()

elapsed = time.perf_counter() - start

print(f"Processes: {elapsed:.2f}s")

Processes: 0.16s


## Massive parellel processing

When you have a single massive array (say $100\text{M}$ elements) and you want to compute on it using all CPU cores in Python, you face a classic problem:Naïve Multiprocessing: If you slice the array and pass chunks to multiprocessing.Pool, Python copies and serializes (pickles) each chunk over OS pipes to every worker process. The serialization overhead is often slower than just running it on a single core.Threading: Threads share the same memory, but the GIL blocks them from executing pure Python bytecode on multiple cores simultaneously.

The 2 Correct Architecture Patterns

Pattern 1: Multiprocessing + Shared Memory (Zero-Copy Slicing)
┌──────────────────────────────────────────────────────────────┐
│ Single Shared Memory Buffer (e.g. 100M items)                │
└──────┬───────────────────┬───────────────────┬───────────────┘
       │ Worker 0          │ Worker 1          │ Worker 2 (Pointers/Offsets only)
       ▼ (Range 0-25M)     ▼ (Range 25-50M)    ▼ (Range 50-75M)
   [ Core 0 ]          [ Core 1 ]          [ Core 2 ]

Pattern 2: C-Extension / OpenMP / NumPy (GIL Release)
   Python Thread 0 ──► calls C / NumPy op ──┐
   Python Thread 1 ──► calls C / NumPy op ──┼──► GIL Released! Native OpenMP executes
   Python Thread 2 ──► calls C / NumPy op ──┘     across all cores at hardware speed.

Pattern 1: Pure Python with multiprocessing.shared_memory (Zero-Copy Work Chunking)
Instead of sending the data to workers, you:

Allocate one shared memory block.

Assign each worker an index offset range (start_idx, end_idx).

Workers attach to the same memory segment and mutate their slice in place.

Here is the exact implementation:

In [12]:
import multiprocessing as mp
from multiprocessing import shared_memory
import numpy as np
import time

def worker(shm_name, shape, dtype, start, end):
    
    exisiting_shm = shared_memory.SharedMemory(name = shm_name)
    
    arr = np.ndarray(shape, dtype=dtype, buffer = existing_shm.buf)
    # This creates a NumPy view over the shared memory.

    # This is important:

    # It doesn't copy the data. ->> ndarray()
    arr[start:end] = arr[start:end]**2
    
    exisiting_shm.close()

def parellel_compute():
    N=20_000_000
    dtype = np.float64
    
    num_cores = mp.cpu_count()
    
    chunk_size = N//num_cores
    
    memory_size = N*np.dtype(dtype).itemsize
    
    shm = shared_memory.SharedMemory(create = True, size = memory_size)
    
    master_arr = np.ndarray((N,), dtype=dtype, buffer = shm.buf)
    
    master_arr[:] = np.arange(N, dtype = dtype)
    
    print(f"Data allocated in shared memory ({memory_size / 1e6:.1f} MB). Cores: {num_cores}")
    
    processes = []
    start_time = time.perf_counter()
    for i in range(num_cores):
        
        start = i*chunk_size
        end = N if i==num_cores-1 else (i+1) * chunk_size
        
        p = mp.Process(target = worker, args = (shm.name, (N,),dtype, start, end))
        processes.append(p)
        p.start()
        
    for p in processes:
        p.join()
    end_time = time.perf_counter()
    print(f"completed in {end_time-start_time:.4f}s")
    
    shm.close()
    shm.unlink()

In [13]:
parellel_compute()

Data allocated in shared memory (160.0 MB). Cores: 14
completed in 0.7580s


Pattern 2: Multi-Threading via C Extensions (Bypassing the GIL)
If your computation is written in C++ (via pybind11, Cython, or C), you explicitly release the GIL using Py_BEGIN_ALLOW_THREADS.

Once the GIL is released, standard Python threads or OpenMP pragmas #pragma omp parallel for automatically light up every CPU core simultaneously with zero memory copying. (We will write this in Pillar 7 / C++ Module).

Key Takeaway for Large Arrays
Never pass large arrays directly to mp.Pool.map (pickling overhead will destroy your throughput).

Use Shared Memory (multiprocessing.shared_memory) and pass index slices [start:end] to each worker.

Each worker reads and writes in-place into its assigned slice of the shared buffer without locks (since memory regions do not overlap).